# Deploy Kanana-2-30B-A3B Model Package from AWS Marketplace 


This sample notebook shows you how to deploy [Kanana-2-30B-A3B](https://aws.amazon.com/marketplace/pp/prodview-wl53qai7vstlq) using Amazon SageMaker.

> **Note**: This is a reference notebook and it cannot run unless you make changes suggested in the notebook.

## Pre-requisites:
1. **Note**: This notebook contains elements which render correctly in Jupyter interface. Open this notebook from an Amazon SageMaker Notebook Instance or Amazon SageMaker Studio.
1. Ensure that IAM role used has **AmazonSageMakerFullAccess**
1. To deploy this ML model successfully, ensure that:
    1. Either your IAM role has these three permissions and you have authority to make AWS Marketplace subscriptions in the AWS account used: 
        1. **aws-marketplace:ViewSubscriptions**
        1. **aws-marketplace:Unsubscribe**
        1. **aws-marketplace:Subscribe**  
    2. or your AWS account has a subscription to [Kanana-2-30B-A3B](https://aws.amazon.com/marketplace/pp/prodview-wl53qai7vstlq). If so, skip step: [Subscribe to the model package](#1.-Subscribe-to-the-model-package)

## Contents:
1. [Subscribe to the model package](#1.-Subscribe-to-the-model-package)
2. [Create an endpoint and perform real-time inference](#2.-Create-an-endpoint-and-perform-real-time-inference)
   1. [Create an endpoint](#A.-Create-an-endpoint)
   2. [Perform real-time inference](#B.-Perform-real-time-inference)
   3. [Perform real-time inference (cli)](#C.-Perform-real-time-inference-(cli))
   4. [Tool-call example of real-time inference](#D.-Tool-call-example-of-real-time-inference)
   5. [Delete the endpoint](#E.-Delete-the-endpoint)
3. [Perform batch inference](#3.-Perform-batch-inference) 
4. [Clean-up](#4.-Clean-up)
    1. [Delete the model](#A.-Delete-the-model)
    2. [Unsubscribe to the listing (optional)](#B.-Unsubscribe-to-the-listing-(optional))
    

## Usage instructions
You can run this notebook one cell at a time (By using Shift+Enter for running a cell).

## 1. Subscribe to the model package

To subscribe to the model package:
1. Open the model package listing page [Kanana-2-30B-A3B](https://aws.amazon.com/marketplace/pp/prodview-wl53qai7vstlq).
1. On the AWS Marketplace listing, click on the **Continue to subscribe** button.
1. On the **Subscribe to this software** page, review and click on **"Accept Offer"** if you and your organization agrees with EULA, pricing, and support terms. 
1. Once you click on **Continue to configuration button** and then choose a **region**, you will see a **Product Arn** displayed. This is the model package ARN that you need to specify while creating a deployable model using Boto3. Copy the ARN corresponding to your region and specify the same in the following cell.

In [ ]:
model_package_arn = "<Customer to specify Model package ARN corresponding to their AWS region>"

In [ ]:
import base64
import json
import uuid
from sagemaker import ModelPackage
import sagemaker as sage
from sagemaker import get_execution_role
from sagemaker import ModelPackage
import boto3
from IPython.display import Image
from PIL import Image as ImageEdit
import numpy as np

In [ ]:
role = get_execution_role()

sagemaker_session = sage.Session()

bucket = sagemaker_session.default_bucket()
runtime = boto3.client("runtime.sagemaker")
bucket

## 2. Create an endpoint and perform real-time inference

If you want to understand how real-time inference with Amazon SageMaker works, see [Documentation](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-hosting.html).

In [ ]:
model_name = "kanana-2-30b-a3b-instruct"

content_type = "application/json"

real_time_inference_instance_type = (
    "ml.g5.12xlarge"
)
batch_transform_inference_instance_type = (
    "ml.g5.12xlarge"
)

### A. Create an endpoint

In [ ]:
# create a deployable model from the model package.
model = ModelPackage(
    role=role, model_package_arn=model_package_arn, sagemaker_session=sagemaker_session
)

# Deploy the model
predictor = model.deploy(1, real_time_inference_instance_type, endpoint_name=model_name)

Once endpoint has been created, you would be able to perform real-time inference.

### B. Perform real-time inference

In [ ]:
def invoke_endpoint(endpoint_name, payload, client):
    """엔드포인트에 추론 요청을 보내고 결과를 반환합니다."""
    response = client.invoke_endpoint(
        EndpointName=endpoint_name,
        Accept="application/json",
        ContentType="application/json",
        Body=json.dumps(payload).encode("utf-8"),
    )
    return json.loads(response["Body"].read().decode("utf-8"))

In [ ]:
# 테스트 1: 기본 추론 (한국어)
payload_basic = {
    "messages": [
        {"role": "user", "content": "한국의 수도는 어디인가요? 간단하게 답변해주세요."}
    ],
    "max_new_tokens": 256,
}

smr_client_ep = sage.local.LocalSagemakerRuntimeClient()
result = invoke_endpoint(model_name, payload_basic, smr_client_ep)

print("=== 기본 추론 (한국어) ===")
print(json.dumps(result, ensure_ascii=False, indent=2))


=== 기본 추론 (한국어) ===
{
  "id": "chatcmpl-139857806733536",
  "object": "chat.completion",
  "created": 1780384869,
  "choices": [
    {
      "index": 0,
      "message": {
        "role": "assistant",
        "content": "서울입니다."
      },
      "logprobs": null,
      "finish_reason": "eos_token"
    }
  ],
  "usage": {
    "prompt_tokens": 18,
    "completion_tokens": 4,
    "total_tokens": 22
  }
}

### C. Perform real-time inference (cli)

In [ ]:
file_name = "real-time-inference-input.json"
output_file_name = "real-time-inference-output.json"

In [ ]:
!aws sagemaker-runtime invoke-endpoint --endpoint-name $model_name --body fileb://$file_name --content-type $content_type --region $sagemaker_session.boto_region_name $output_file_name

### D. Tool-call example of real-time inference 

In [ ]:
tools = [{
    "type": "function",
    "function": {
        "name": "getCurrentTimeForLocation",
        "description": "특정 지역의 현재 시간 정보 제공",
        "parameters": {
            "type": "object",
            "properties": {
                "location": {"type": "string", "description": "시간 정보를 가져올 지역 이름"}
            },
            "required": ["location"]
        }
    }
}]

payload_tool = {
    "messages": [
        {"role": "user", "content": "Honolulu 지금 몇시야?"}
    ],
    "max_new_tokens": 256,
    "temperature": 0.1,
    "tools": tools,
    "tool_choice": "auto"
}


result = invoke_endpoint(model_name, payload_tool, smr_client_ep)
print("=== 툴콜 추론 ===")
print(json.dumps(result, ensure_ascii=False, indent=2))

### E. Delete the endpoint

Now that you have successfully performed a real-time inference, you do not need the endpoint any more. You can terminate the endpoint to avoid being charged.

In [ ]:
model.sagemaker_session.delete_endpoint(model_name)
model.sagemaker_session.delete_endpoint_config(model_name)

## 3. Perform batch inference

In this section, you will perform batch inference using multiple input payloads together. If you are not familiar with batch transform, and want to learn more, see these links:
1. [How it works](https://docs.aws.amazon.com/sagemaker/latest/dg/ex1-batch-transform.html)
2. [How to run a batch transform job](https://docs.aws.amazon.com/sagemaker/latest/dg/how-it-works-batch.html)

In [ ]:
# upload the batch-transform job input files to S3
transform_input_folder = "data/input/batch"
transform_input = sagemaker_session.upload_data(transform_input_folder, key_prefix=model_name)
print("Transform input uploaded to " + transform_input)

In [ ]:
# Run the batch-transform job
transformer = model.transformer(1, batch_transform_inference_instance_type)
transformer.transform(transform_input, content_type=content_type)
transformer.wait()

In [ ]:
# output is available on following path
transformer.output_path

example of batch transform output:

{"id": "chatcmpl-140174525960256", "object": "chat.completion", "created": 1780387099, "choices": [{"index": 0, "message": {"role": "assistant", "content": "한국의 수도는 서울입니다."}, "logprobs": null, "finish_reason": "eos_token"}], "usage": {"prompt_tokens": 14, "completion_tokens": 32, "total_tokens": 46}}

## 4. Clean-up

### A. Delete the model

In [ ]:
model.delete_model()

### B. Unsubscribe to the listing (optional)

If you would like to unsubscribe to the model package, follow these steps. Before you cancel the subscription, ensure that you do not have any [deployable model](https://console.aws.amazon.com/sagemaker/home#/models) created from the model package or using the algorithm. Note - You can find this information by looking at the container name associated with the model. 

**Steps to unsubscribe to product from AWS Marketplace**:
1. Navigate to __Machine Learning__ tab on [__Your Software subscriptions page__](https://aws.amazon.com/marketplace/ai/library?productType=ml&ref_=mlmp_gitdemo_indust)
2. Locate the listing that you want to cancel the subscription for, and then choose __Cancel Subscription__  to cancel the subscription.

